# Compute exposure time

## using SPOCK ETC

In [77]:
import SPOCK.ETC as ETC

# ------- test  ETC

ETC_vec = (ETC.etc(mag_val= 13.11 , mag_band='J', spt='M9',filt='I+z',  airmass=1.3, moonphase=0.5, irtf=0.8, num_tel=1, seeing=1.5, gain=1.1))#airmass=1.1, moonphase=0.5, irtf=0.8, num_tel=1, seeing=2., gain=1.1))
texp = ETC_vec.exp_time_calculator(ADUpeak=50000)[0]

print(texp)


623.7852245756754


## using mphot

In [16]:
import mphot
from SPOCK import path_spock

Teff_target = 2632 #K
gaia_id = 56252256123908096

# files used to generate SR
efficiencyFile2 = path_spock + '/SPOCK/files_ETC/SPIRIT/datafiles/systems/pirtSPC_-60.csv'
filterFile2 = path_spock + '/SPOCK/files_ETC/SPIRIT/datafiles/filters/zYJ.csv'

# name to refer to the generated file
name, system_response = mphot.generate_system_response(
    efficiencyFile2, filterFile2
)

# sky properties
props_sky = {
    "pwv": 2.5,  # PWV [mm]
    "airmass": 1.1,  # airmass
    "seeing": 1.,  # seeing (==FWHM) ["]
}
props_callisto = {
    "name": name,
    "plate_scale": 0.35 * (12 / 13.5),
    "N_dc": 230,
    "N_rn": 80,
    "well_depth": 55000,
    "bias_level": 0,
    "well_fill": 0.7,
    "read_time": 0.1,
    "r0": 0.5,
    "r1": 0.14,
    "ap_rad": 3
}


INFO:mphot.core:`/Users/ed268546/env_SPOCKv2/lib/python3.12/site-packages/mphot/datafiles/system_responses/pirtSPC_-60_zYJ_instrument_system_response.csv` has been generated and saved!


In [17]:
# get the precision and components used to calculate it (generates grid if not already present)
result = mphot.get_precision_gaia(
    props_callisto, props_sky, source_id=gaia_id, Teff=Teff_target
)

mphot.display_results(result)


INFO:astroquery:Query finished.


INFO: Query finished. [astroquery.utils.tap.core]


,single frame [ppt],10 minute binned [ppt]
,pirtSPC_-60_zYJ,pirtSPC_-60_zYJ
All,4.26,0.529
Star,1.51,0.188
Scintillation,1.39,0.172
Sky,0.991,0.123
Dark current,1.79,0.222
Read noise,3.13,0.388


,pirtSPC_-60_zYJ
Teff [K],2.63e+3
distance [pc],14.6
N_star [e/s],4.79e+4
star_flux [e/m2/s],6.62e+4
scn [e_rms],607
pixels in aperture [pix],292
ap_radius [pix],9.64
N_sky [e/pix/s],70.5
sky_radiance [e/m2/arcsec2/s],1.01e+3
seeing [arcsec],1.00


,pirtSPC_-60_zYJ
star [mag],12.5
sky [mag/arcsec2],17.1
vega_flux [e/s],4.94e+9


In [ ]:
# extract exposure time
image_precision, binned_precision, components = result

exposure_time = components["t [s]"]
int(exposure_time)

9

# Long term scheduler 

In [2]:
import SPOCK.long_term_scheduler as SPOCKLT

#---- Long-term scheduler

schedule = SPOCKLT.Schedules()
schedule.observatory_name = 'SSO'
schedule.telescope = 'Io'
schedule.load_parameters(date_range=['2025-11-04 15:00:00', '2025-11-05 15:00:00'])
schedule.make_schedule(Altitude_constraint=28, Moon_constraint=30)
SPOCKLT.save_schedule(save=True, over_write=True, date_range=schedule.date_range, telescope=schedule.telescope)
SPOCKLT.make_plans(day=schedule.date_range[0], nb_days=schedule.date_range_in_days, telescope=schedule.telescope)


Updating hours of obs :   0%|          | 0/1 [00:00<?, ?it/s]

Scheduling :   0%|          | 0/1 [00:00<?, ?it/s]

INFO:  day is :  2025-11-04 15:00:00.000


KeyboardInterrupt: 

In [ ]:
SPOCKLT.upload_plans(day=schedule.date_range[0], nb_days=schedule.date_range_in_days, telescope=schedule.telescope)


# Short term scheduler

In [58]:
import SPOCK.short_term_scheduler as SPOCKST

# ---- Short-term scheduler
schedule = SPOCKST.Schedules()
schedule.load_parameters()
schedule.load_parameters()
schedule.day_of_night = Time('2025-09-23 15:00:00')
schedule.observatory_name = 'SSO'
schedule.telescope = 'Io'
schedule.special_target(input_name="Trappist-1")
schedule.make_scheduled_table()
schedule.planification()
schedule.make_night_block()
SPOCKST.save_schedule(save=True, over_write=True, day=schedule.day_of_night,
                              telescope=schedule.telescope)
SPOCKST.make_plans(day=schedule.day_of_night, nb_days=1,
                          telescope=schedule.telescope)
# SPOCKST.upload_plans(day=schedule.day_of_night, nb_days=1,
#                             telescope = schedule.telescope)


INFO:  Not using moon phase in ETC

INFO:  Local path exists and is:  /Archive_night_blocks/night_blocks_Io_2025-09-23.txt
INFO:  Local path exists and is:  /Archive_night_blocks/night_blocks_Io_2025-09-23.txt
INFO:  situation 9
INFO:  situation 2
INFO:  no transition block
INFO:  "/Users/ed268546/Documents/codes/SPOCK_v2/night_blocks_propositions/night_blocks_Io_2025-09-23.txt" has been over-written to "/Users/ed268546/Documents/codes/SPOCK_v2/DATABASE/Io/"
INFO:  Path exists and is:  /Users/ed268546/Documents/codes/SPOCK_v2/DATABASE/Io/night_blocks_Io_2025-09-23.txt

INFO:  Path exists and is:  /Users/ed268546/Documents/codes/SPOCK_v2/DATABASE/Io/night_blocks_Io_2025-09-23.txt

INFO: Target Sp2047+1421 has been scheduled on the 2025-09-23 on Io
INFO: Target dome_rot has been scheduled on the 2025-09-23 on Io
INFO: Target Trappist-1 has been scheduled on the 2025-09-23 on Io
INFO: Target Sp0246+1625 has been scheduled on the 2025-09-23 on Io
